<a id="top"></a>
# `add_asdf` Tutorial

In this tutorial, we show how to load Roman asdf files into `mast-aladin`. We leverage `mast-table` to display `astroquery` results and identify the products of interest.

<div class="alert alert-warning">
Warning: the following cells rely on Roman test data. If a user lacks the proper data permissions, these cells will NOT run.
</div>

### Imports

In [ ]:
from mast_aladin import MastAladin
from mast_table import MastTable
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.mast import MastMissions, conf

# configure MastMissions to be able to query Roman test data
conf.server = "https://masttest.stsci.edu"
mast = MastMissions(mission='roman')
mast._service_api_connection.SERVICE_URL = 'https://masttest.stsci.edu'
mast._service_api_connection.REQUEST_URL = 'https://masttest.stsci.edu/search/roman/api/v0.1/'
mast._service_api_connection.MISSIONS_DOWNLOAD_URL = 'https://masttest.stsci.edu/search/'

# update to use your own token from:
# https://auth.masttest.stsci.edu/token?suggested_name=Astroquery&suggested_scope=mast:exclusive_access
user_token = "your_masttest_token_here"
my_session = mast.login(token=user_token)

### Query observations

In [ ]:
# query in the region of the Cat's Eye Nebula (NGC 6543)
target = SkyCoord(ra=269.8325254855945, dec=66.03550955132157, unit='deg')
obs = mast.query_region(target, radius=0.01 * u.deg)

# load into mast-table to inspect the returned observations and display the table
mast_table_obs = MastTable(obs)
mast_table_obs

In [ ]:
# now, we filter for the product we are interested in
mast_table_obs.selected_rows = mast_table_obs.items[0]

### Query products

In [ ]:
# using the selection from above, determine the products associated with the selected observation
products = mast.get_product_list(mast_table_obs.selected_rows_table)
cal_products = mast.filter_products(products, file_suffix='_cal')

# load into mast-table to inspect the available products and display the table
mast_table_products = MastTable(cal_products)
mast_table_products

In [ ]:
# now, filter for the product we are interested in
mast_table_products.selected_rows = mast_table_products.items[0]

In [ ]:
# lastly, download this file to read into MastAladin
download_filepath = mast.download_products(mast_table_products.selected_rows_table)['Local Path']

### Use `add_asdf` to display files in `mast-aladin`

In [ ]:
# create a mast-aladin instance and display
mast_aladin = MastAladin(
    fov = 1
)
mast_aladin

In [ ]:
# add the outline of the expected layer
s_region = "POLYGON ICRS 269.987238 65.9839542 269.987238 66.1068995 269.676782731 66.1068995 269.680114248 65.9839542"
layer_outline = mast_aladin.add_graphic_overlay_from_stcs(s_region, name = "asdf outline")

# pass the filepath to the asdf we want to add
asdf_filepath = "./"+str(download_filepath[0])

mast_aladin.add_asdf(asdf_filepath)

In [ ]:
# we can pass other arguments related to the image options, including name and colormap for example
# for a full account of possible arguments, see: https://cds-astro.github.io/aladin-lite/global.html#ImageOptions
mast_aladin.add_asdf(asdf_filepath, name = "Roman Build 19 Test ASDF", colormap = "viridis")

## About this notebook

**Last updated:** October 2026
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 